# Day 18 — Capstone Project: RAG Agent
## 30 Days of AI: From NLP to LLMs

---

This is your first capstone. Every concept from Days 11–17 comes
together into one production-ready system:

```
Day 11  :  LLM fundamentals — API calls, tokens, temperature
Day 12  :  Prompt engineering — system prompts, CoT, JSON output
Day 13  :  Embeddings — semantic search, cosine similarity
Day 14  :  RAG retrieval — chunking, vector store, evaluation
Day 15  :  RAG generation — prompts, citation, grounding, refusal
Day 16  :  LangChain — LCEL, memory, chains
Day 17  :  Agents — ReAct loop, tools, function calling
```

What you will build today is a **RAG Agent** — an agent that:
1. Maintains conversation memory across turns
2. Has a vector store of documents as a searchable knowledge base
3. Has additional tools: calculator, text analyzer, metadata filter
4. Decides autonomously when to search docs vs calculate vs reason
5. Cites its sources and refuses to answer from outside the corpus
6. Evaluates its own answers for faithfulness

---

### Project Specification

```
System: AI Research Assistant
Corpus: 8 AI/ML documents
Tools :
  search_documents   →  semantic search over the vector store
  calculator         →  safe math evaluation
  get_document_list  →  list all available documents
  summarize_document →  fetch and summarize a specific document
  compare_concepts   →  retrieve chunks for two topics, compare

Memory : ConversationBufferMemory (last 10 turns)
Eval   : faithfulness + relevance scoring on every answer
```

### Goal by End of Day

A fully working, end-to-end RAG Agent with a clean
interactive Q&A loop, source citations, automatic
evaluation, and a summary report of session quality.

In [ ]:
## Run once
## !pip install langchain langchain-openai langchain-anthropic \
##             langchain-community sentence-transformers faiss-cpu -q

import os, re, json, math, time, hashlib, warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from collections import defaultdict

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
%matplotlib inline
warnings.filterwarnings('ignore')

from sentence_transformers import SentenceTransformer

try:
    import faiss
    FAISS = True
except ImportError:
    FAISS = False

HAS_OPENAI    = bool(os.environ.get('OPENAI_API_KEY'))
HAS_ANTHROPIC = bool(os.environ.get('ANTHROPIC_API_KEY'))

def call_llm(prompt, system='You are a helpful assistant.',
             temperature=0.1, max_tokens=600):
    if HAS_OPENAI:
        from openai import OpenAI
        c = OpenAI()
        r = c.chat.completions.create(
            model='gpt-3.5-turbo',
            messages=[{'role':'system','content':system},
                      {'role':'user','content':prompt}],
            temperature=temperature, max_tokens=max_tokens)
        return r.choices[0].message.content
    elif HAS_ANTHROPIC:
        import anthropic
        c = anthropic.Anthropic()
        r = c.messages.create(
            model='claude-3-haiku-20240307', max_tokens=max_tokens,
            system=system,
            messages=[{'role':'user','content':prompt}])
        return r.content[0].text
    else:
        return ('[MOCK] Based on the retrieved context, here is the answer. '
                'The documents describe this concept clearly. (Source: doc.txt)')

provider = 'openai' if HAS_OPENAI else 'anthropic' if HAS_ANTHROPIC else 'mock'
print(f'Provider : {provider}')
print('All imports ready.')

In [ ]:
# ================================================================
# STEP 1 — BUILD THE KNOWLEDGE BASE
# 8 AI/ML documents covering the full course so far
# ================================================================

@dataclass
class Document:
    text     : str
    metadata : Dict = field(default_factory=dict)

@dataclass
class Chunk:
    text     : str
    metadata : Dict = field(default_factory=dict)
    chunk_id : str  = ''
    def __post_init__(self):
        if not self.chunk_id:
            self.chunk_id = hashlib.md5(self.text.encode()).hexdigest()[:8]

DOCUMENTS = [
    Document(
        text="""Supervised Learning Fundamentals

Supervised learning trains models on labeled input-output pairs.
The model learns a mapping f(X) → Y that minimises a loss function.
At prediction time it applies this mapping to unseen inputs.

Classification predicts discrete categories. Regression predicts
continuous values. Decision boundaries separate classes in feature space.

The train/validation/test split is essential. Train on 70%, tune
hyperparameters on 10% validation, report final accuracy on held-out
test set. Never use test data during development.

Key algorithms: logistic regression (linear boundary), decision trees
(axis-aligned splits), random forests (ensemble of trees), SVMs
(maximum margin boundary), gradient boosting (XGBoost, LightGBM).

Overfitting: model memorises training noise. Prevents generalisation.
Fixes: regularisation (L1 shrinks weights to zero, L2 penalises large
weights), dropout (randomly zero neurons during training), early
stopping (halt when validation loss stops improving).""",
        metadata={'source':'supervised_learning.txt', 'topic':'machine_learning', 'difficulty':'beginner'}
    ),
    Document(
        text="""Neural Networks and Backpropagation

A neural network is a directed acyclic graph of parameterised
linear transformations followed by non-linear activations.

Forward pass: input → hidden layers → output logits → loss.
Backward pass: chain rule computes gradient of loss with respect
to every weight. Gradient descent updates weights in the direction
that reduces loss.

Activation functions introduce non-linearity. ReLU (max(0,x)) is
the default for hidden layers — avoids vanishing gradients, fast.
Sigmoid squashes to (0,1) for binary output. Softmax normalises
a vector to a probability distribution for multi-class output.

Batch normalisation normalises activations within each mini-batch.
Reduces internal covariate shift. Allows higher learning rates.
Placed after linear layer, before activation.

Adam optimiser combines momentum and adaptive learning rates.
It is the default choice for training neural networks. Key
hyperparameters: learning rate (1e-3 default), beta1=0.9, beta2=0.999.""",
        metadata={'source':'neural_networks.txt', 'topic':'deep_learning', 'difficulty':'intermediate'}
    ),
    Document(
        text="""Transformers and Attention

The Transformer (Vaswani et al., 2017) replaces recurrence with
self-attention. All positions are processed in parallel, enabling
GPU-efficient training on long sequences.

Scaled dot-product attention: Attention(Q,K,V) = softmax(QKᵀ/√d_k)V.
Q, K, V are linear projections of the input. The denominator √d_k
prevents softmax saturation when dimensions are large.

Multi-head attention runs h attention functions in parallel, each
on a lower-dimensional subspace. Outputs are concatenated and
projected. Allows attending to multiple relationship types simultaneously.

Positional encoding adds position information since self-attention
is permutation-invariant. Sinusoidal encoding uses sin/cos waves
of different frequencies. Learned positional embeddings are common
in BERT and GPT.

BERT (encoder-only, bidirectional) excels at classification and NER.
GPT (decoder-only, causal) excels at text generation.
T5 (encoder-decoder) excels at seq2seq tasks like translation.""",
        metadata={'source':'transformers.txt', 'topic':'nlp', 'difficulty':'intermediate'}
    ),
    Document(
        text="""Large Language Models

LLMs are transformer-based models trained on internet-scale text
corpora using next-token prediction as the self-supervised objective.
GPT-3 (175B params, 2020) demonstrated emergent few-shot abilities.

Training pipeline:
1. Pretraining: predict next token over trillions of web tokens.
2. Supervised Fine-Tuning (SFT): train on human-written (prompt, response) pairs.
3. RLHF: reward model trained on human preference rankings, then PPO
   optimises the policy to maximise reward while staying close to SFT.

Context window is the maximum tokens processed in one call.
GPT-4 supports 128K tokens. Claude supports 200K tokens.
Attention is O(n²) in sequence length — long contexts are expensive.

Temperature T controls generation diversity. T→0 is deterministic.
T>1 flattens the distribution for more creative outputs.
Top-p nucleus sampling keeps the smallest token set summing to p.

Fine-tuning adapts a pretrained LLM to a domain. LoRA adds trainable
low-rank matrices (rank 4-64) to attention projections. Only ~0.1-1%
of parameters are updated. Full weights are preserved and frozen.""",
        metadata={'source':'large_language_models.txt', 'topic':'nlp', 'difficulty':'intermediate'}
    ),
    Document(
        text="""RAG — Retrieval-Augmented Generation

RAG (Lewis et al., 2020) grounds LLM generation in retrieved documents.
It solves two fundamental LLM problems: knowledge cutoff and hallucination.

Offline pipeline: load documents → chunk (300-500 tokens, 10% overlap)
→ embed each chunk → store in vector database.

Online pipeline: embed query → retrieve top-k chunks by cosine
similarity → inject chunks into LLM prompt → generate grounded answer.

Chunking strategy matters greatly. Recursive character splitting
preserves paragraph and sentence boundaries. Overlapping chunks
prevent information loss at boundaries.

Advanced retrieval: HyDE generates a hypothetical answer and embeds
it for retrieval (better query-document alignment). Cross-encoder
re-ranking rescores top-50 candidates for higher precision.

RAG evaluation: context recall (are all needed facts retrieved?),
faithfulness (does the answer only use retrieved context?),
answer relevance (does the answer address the question?).

The RAG system should refuse to answer when retrieved context
has low similarity scores or does not contain the answer.
Silence is better than confident hallucination.""",
        metadata={'source':'rag_system.txt', 'topic':'nlp', 'difficulty':'intermediate'}
    ),
    Document(
        text="""Prompt Engineering

Prompt engineering is the practice of crafting LLM inputs to
reliably elicit desired outputs without changing model weights.

Anatomy of an effective prompt:
  System : role, constraints, output format
  Context: relevant background information
  Examples: few-shot input/output demonstrations
  Instruction: the specific task to perform
  Format: how the output should be structured

Chain-of-Thought (Wei et al., 2022): asking the model to explain
reasoning steps before answering dramatically improves accuracy
on multi-step reasoning tasks. 'Let's think step by step.'

Self-consistency: sample CoT reasoning N times at temperature>0.
Take majority vote on final answers. Improves reliability.

Few-shot prompting provides 3-8 input/output examples before
the target query. Demonstrates format, edge cases, and style.
More reliable than zero-shot for custom tasks.

Common failure modes: hallucination (fix: ground in context),
format non-compliance (fix: explicit format + 'ONLY return JSON'),
sycophancy (fix: 'if my premise is wrong, say so explicitly').""",
        metadata={'source':'prompt_engineering.txt', 'topic':'llm_application', 'difficulty':'beginner'}
    ),
    Document(
        text="""LangChain Framework

LangChain is an open-source framework for building LLM applications.
It provides composable abstractions for the most common patterns.

LCEL (LangChain Expression Language) uses the | pipe operator to
compose chains: prompt | llm | output_parser. Each step's output
is the next step's input. Supports streaming and batching.

Key components:
  ChatOpenAI / ChatAnthropic : model wrappers with unified interface
  PromptTemplate             : parameterised prompt factories
  StrOutputParser            : extract .content from AIMessage
  FAISS.from_documents()     : create vector store from documents
  ConversationBufferMemory   : store and inject full chat history
  ConversationSummaryMemory  : compress old turns to save tokens
  AgentExecutor              : ReAct loop with tools and memory

When to use LangChain: production systems, rapid iteration,
teams, integrations with many providers and tools.
When to build manually: learning, maximum control, minimal
dependencies, custom architecture.""",
        metadata={'source':'langchain_guide.txt', 'topic':'frameworks', 'difficulty':'intermediate'}
    ),
    Document(
        text="""AI Agents and Tool Use

An agent is an LLM in a reasoning-action loop that uses tools
to accomplish tasks that require multiple steps or external data.

ReAct (Yao et al., 2022) interleaves Thought, Action, and Observation.
The LLM reasons, selects a tool, runs it, observes the result,
and reasons again until it reaches a Final Answer.

Function calling (OpenAI/Anthropic API feature) is more reliable
than text-based ReAct. Tools are described in a JSON schema.
The model outputs a structured JSON tool call rather than free text.
This eliminates text parsing failures.

Tool design best practices:
  1. Write precise tool descriptions — LLM reads these to decide
  2. Specify exact input format with examples
  3. Include when NOT to use each tool
  4. Wrap all logic in try/except — return error strings
  5. Keep tools focused: one clear purpose each

Agent failure modes: infinite loops (fix: max_iterations),
hallucinated tool inputs (fix: input validation + clear description),
wrong tool selected (fix: clearer descriptions with negative examples).

Combining RAG + agents: make the vector store a tool called
search_documents. The agent decides when to search vs calculate
vs reason from existing context. This is a RAG Agent.""",
        metadata={'source':'ai_agents.txt', 'topic':'agents', 'difficulty':'intermediate'}
    ),
]

print(f'Knowledge base: {len(DOCUMENTS)} documents')
for doc in DOCUMENTS:
    print(f'  {doc.metadata["source"]:<35} topic={doc.metadata["topic"]}')

In [ ]:
# ================================================================
# STEP 2 — VECTOR STORE
# Chunk, embed, and index all documents
# ================================================================

def recursive_chunker(doc, chunk_size=350, overlap=50):
    seps = ['\n\n', '\n', '. ', ' ', '']
    def split(text, si=0):
        if len(text) <= chunk_size or si >= len(seps): return [text] if text.strip() else []
        sep, parts, merged, cur = seps[si], text.split(seps[si]) if seps[si] else list(text), [], ''
        for p in parts:
            cand = (cur + sep + p).strip() if cur else p.strip()
            if len(cand) <= chunk_size: cur = cand
            else:
                if cur: merged.append(cur)
                cur = p.strip() if len(p) <= chunk_size else (merged.extend(split(p, si+1)) or '')
        if cur: merged.append(cur)
        return [m for m in merged if m.strip()]
    raw = split(doc.text)
    chunks = []
    for i, t in enumerate(raw):
        if i > 0: t = raw[i-1][-overlap:] + ' ' + t
        chunks.append(Chunk(text=t.strip(), metadata={**doc.metadata, 'chunk_index': i}))
    return chunks


class VectorStore:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model  = SentenceTransformer(model_name)
        self.dim    = self.model.get_sentence_embedding_dimension()
        self.chunks = []
        if FAISS:
            self.index = faiss.IndexFlatIP(self.dim)
        else:
            self._vecs = None

    def ingest(self, docs, chunk_size=350, overlap=50):
        all_chunks = [c for d in docs for c in recursive_chunker(d, chunk_size, overlap)]
        vecs = self.model.encode([c.text for c in all_chunks],
                                  normalize_embeddings=True,
                                  show_progress_bar=True).astype(np.float32)
        self.chunks.extend(all_chunks)
        if FAISS: self.index.add(vecs)
        else: self._vecs = vecs if self._vecs is None else np.vstack([self._vecs, vecs])
        print(f'  Indexed {len(self.chunks)} chunks.')

    def search(self, query, top_k=4, topic=None):
        q = self.model.encode([query], normalize_embeddings=True).astype(np.float32)
        k = top_k * 4 if topic else top_k
        if FAISS:
            scores, ids = self.index.search(q, k)
            scores, ids = scores[0], ids[0]
        else:
            all_s = self._vecs @ q[0]
            ids   = np.argsort(all_s)[::-1][:k]
            scores = all_s[ids]
        results = []
        for s, i in zip(scores, ids):
            if i < 0 or i >= len(self.chunks): continue
            c = self.chunks[i]
            if topic and c.metadata.get('topic') != topic: continue
            results.append({'score': float(s), 'text': c.text, 'metadata': c.metadata})
            if len(results) == top_k: break
        return results

    def __len__(self): return len(self.chunks)


print('Building knowledge base vector store...')
VS = VectorStore()
VS.ingest(DOCUMENTS)
print(f'\nVector store ready: {len(VS)} chunks from {len(DOCUMENTS)} documents.')

In [ ]:
# ================================================================
# STEP 3 — AGENT TOOLS
# 5 tools the agent can use to answer questions
# ================================================================

from langchain_core.tools import tool

@tool
def search_documents(query: str) -> str:
    """
    Semantically searches the AI/ML knowledge base.
    Use this for ANY question about AI, ML, NLP, transformers, RAG,
    agents, LLMs, prompt engineering, or LangChain.
    Input: a clear natural language question or keyword phrase.
    Returns: the most relevant passages with source citations.
    """
    results = VS.search(query, top_k=3)
    if not results:
        return 'No relevant documents found for this query.'
    parts = []
    for r in results:
        parts.append(f"[Source: {r['metadata']['source']} | Score: {r['score']:.3f}]\n{r['text']}")
    return '\n\n'.join(parts)


@tool
def calculator(expression: str) -> str:
    """
    Evaluates a mathematical expression.
    Use for arithmetic, percentages, and numerical calculations.
    Input: a valid Python math expression like '175e9 * 4 / 1e9' or '0.15 * 2500'.
    Do NOT pass text descriptions — only pure numeric Python expressions.
    """
    try:
        allowed = {k: getattr(math, k) for k in dir(math) if not k.startswith('_')}
        allowed.update({'abs': abs, 'round': round})
        result = eval(expression, {'__builtins__': {}}, allowed)
        return f'{float(result):,.6g}'
    except Exception as e:
        return f'Calculation error: {e}'


@tool
def get_document_list(dummy: str = '') -> str:
    """
    Returns a list of all documents in the knowledge base with their topics.
    Use when the user asks what documents or topics are available.
    Input: any string (input is ignored).
    """
    lines = ['Available documents in the knowledge base:']
    for doc in DOCUMENTS:
        lines.append(
            f"  • {doc.metadata['source']}"
            f" | topic: {doc.metadata['topic']}"
            f" | difficulty: {doc.metadata['difficulty']}"
        )
    return '\n'.join(lines)


@tool
def compare_concepts(concepts: str) -> str:
    """
    Retrieves and compares information about two AI/ML concepts side by side.
    Use when the user asks to compare, contrast, or explain the difference
    between two techniques or models.
    Input: two concepts separated by ' vs ' e.g. 'BERT vs GPT' or 'RAG vs fine-tuning'.
    """
    parts = [p.strip() for p in concepts.lower().split(' vs ')]
    if len(parts) != 2:
        return 'Format: "concept1 vs concept2". Example: "BERT vs GPT"'
    results = {}
    for concept in parts:
        hits = VS.search(concept, top_k=2)
        results[concept] = hits[0]['text'][:300] if hits else 'No info found.'
    return (
        f'--- {parts[0].upper()} ---\n{results[parts[0]]}\n\n'
        f'--- {parts[1].upper()} ---\n{results[parts[1]]}'
    )


@tool
def get_learning_path(topic: str) -> str:
    """
    Suggests a learning path for a given AI/ML topic based on the knowledge base.
    Use when the user asks how to learn something, what to study next,
    or wants a structured approach to a topic.
    Input: the topic or skill the user wants to learn.
    """
    # Search for related documents
    hits = VS.search(f'how to learn {topic}', top_k=3)
    sources = [h['metadata']['source'] for h in hits]
    # Build a simple path recommendation
    difficulty_order = ['beginner', 'intermediate', 'advanced']
    sorted_docs = sorted(
        [d for d in DOCUMENTS if d.metadata['source'] in sources],
        key=lambda d: difficulty_order.index(d.metadata.get('difficulty', 'intermediate'))
    )
    if not sorted_docs:
        return f'No learning path found for "{topic}" in the knowledge base.'
    steps = [f'Step {i+1}: {d.metadata["source"]} ({d.metadata["difficulty"]})"
              for i, d in enumerate(sorted_docs)]
    return f'Suggested learning path for "{topic}":\n' + '\n'.join(steps)


AGENT_TOOLS = [search_documents, calculator, get_document_list,
               compare_concepts, get_learning_path]

print('Agent tools:')
for t in AGENT_TOOLS:
    print(f'  • {t.name}')

In [ ]:
# ================================================================
# STEP 4 — THE RAG AGENT
# LangChain AgentExecutor + memory + tools
# ================================================================

from langchain.agents        import create_react_agent, AgentExecutor
from langchain.memory        import ConversationBufferWindowMemory
from langchain_core.prompts  import PromptTemplate

def get_langchain_llm(temperature=0.1):
    if HAS_OPENAI:
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model='gpt-3.5-turbo', temperature=temperature)
    elif HAS_ANTHROPIC:
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model='claude-3-haiku-20240307', temperature=temperature)
    else:
        from langchain_core.language_models.fake import FakeListChatModel
        return FakeListChatModel(responses=[
            'Thought: I need to search the documents.\nAction: search_documents\nAction Input: overfitting prevention\nObservation: ...\nThought: I have enough information.\nFinal Answer: Overfitting is prevented by regularization and dropout.',
        ] * 50)


RAG_AGENT_SYSTEM = """
You are an expert AI Research Assistant with access to a curated knowledge base
of AI and machine learning documents.

TOOLS AVAILABLE:
{tools}

STRICT RULES:
1. ALWAYS use search_documents for any AI/ML question before answering.
2. ONLY answer using information from tool results. Never use prior knowledge alone.
3. Cite sources explicitly: (Source: filename.txt) after each claim.
4. If no relevant document is found, say: 'This topic is not covered in the knowledge base.'
5. Use calculator for any numerical calculation — never compute mentally.
6. Use compare_concepts when asked to compare two things.

FORMAT (follow exactly):
Thought: <your reasoning>
Action: <tool name from [{tool_names}]>
Action Input: <tool input>
Observation: <tool output — filled by system>
... (repeat as needed)
Thought: I have enough information to answer.
Final Answer: <comprehensive answer with source citations>

Conversation history:
{chat_history}

Question: {input}
Thought:{agent_scratchpad}"""

agent_prompt = PromptTemplate.from_template(RAG_AGENT_SYSTEM)

# Memory: keep last 10 turns (5 exchanges)
agent_memory = ConversationBufferWindowMemory(
    k               = 10,
    memory_key      = 'chat_history',
    return_messages = False,
    input_key       = 'input',
)

# Create agent and executor
react_agent = create_react_agent(
    llm    = get_langchain_llm(temperature=0.1),
    tools  = AGENT_TOOLS,
    prompt = agent_prompt,
)

RAG_AGENT = AgentExecutor(
    agent                 = react_agent,
    tools                 = AGENT_TOOLS,
    memory                = agent_memory,
    verbose               = True,
    max_iterations        = 8,
    handle_parsing_errors = True,
    return_intermediate_steps = True,
)

print('RAG Agent ready.')
print(f'Tools: {[t.name for t in AGENT_TOOLS]}')
print(f'Memory: ConversationBufferWindowMemory (k=10)')

In [ ]:
# ================================================================
# STEP 5 — EVALUATION LAYER
# Automatic faithfulness + relevance scoring on every answer
# ================================================================

@dataclass
class EvalRecord:
    question     : str
    answer       : str
    sources      : List[str]
    tools_used   : List[str]
    faithfulness : float = 0.0
    relevance    : float = 0.0
    latency_ms   : float = 0.0


def evaluate_answer(question, answer, context):
    """Score faithfulness and relevance using LLM-as-judge."""
    judge_prompt = f"""
Rate the following AI assistant answer on two dimensions. Return JSON only.

Context retrieved: {context[:500]}
Question: {question}
Answer: {answer[:400]}

Return ONLY this JSON:
{{"faithfulness": <0.0-1.0, is every claim supported by context?>,
  "relevance": <0.0-1.0, does the answer address the question?>,
  "verdict": "<good|acceptable|poor>"}}"""

    raw = call_llm(judge_prompt, temperature=0, max_tokens=100)
    try:
        raw = re.sub(r'^```(?:json)?\s*', '', raw.strip())
        raw = re.sub(r'\s*```$', '', raw)
        return json.loads(raw)
    except:
        return {'faithfulness': 0.75, 'relevance': 0.75, 'verdict': 'acceptable'}


session_records = []


def ask_agent(question: str) -> EvalRecord:
    """Run the agent, evaluate the answer, store the record."""
    t0 = time.time()

    try:
        result     = RAG_AGENT.invoke({'input': question})
        answer     = result.get('output', 'No answer generated.')
        steps      = result.get('intermediate_steps', [])
        tools_used = [s[0].tool for s in steps if hasattr(s[0], 'tool')]
        # Build context from all search_documents observations
        context    = ' '.join(
            str(s[1]) for s in steps
            if hasattr(s[0], 'tool') and s[0].tool == 'search_documents'
        )[:800]
    except Exception as e:
        answer     = f'Agent error: {str(e)[:100]}'
        tools_used = []
        context    = ''

    latency = (time.time() - t0) * 1000

    # Evaluate
    eval_result  = evaluate_answer(question, answer, context)

    # Extract cited sources from answer text
    sources = re.findall(r'Source:\s*([\w._]+)', answer)

    record = EvalRecord(
        question     = question,
        answer       = answer,
        sources      = list(set(sources)),
        tools_used   = tools_used,
        faithfulness = eval_result.get('faithfulness', 0),
        relevance    = eval_result.get('relevance', 0),
        latency_ms   = latency,
    )
    session_records.append(record)
    return record


print('Evaluation layer ready.')
print('ask_agent(question) → run agent + evaluate + store record')

In [ ]:
# ================================================================
# STEP 6 — RUN THE FULL SESSION
# 8 questions that test different capabilities
# ================================================================

TEST_QUESTIONS = [
    # Tests: search_documents — factual knowledge
    'What is the difference between BERT and GPT in terms of architecture?',

    # Tests: search + calculator — hybrid
    'GPT-3 has 175 billion parameters. If each parameter is stored in float16 '
    '(2 bytes), how many gigabytes is the model?',

    # Tests: compare_concepts
    'Compare RAG vs fine-tuning — when should I use each?',

    # Tests: document_list
    'What topics does the knowledge base cover?',

    # Tests: search — multi-hop reasoning
    'How does RLHF relate to the LLM training pipeline?',

    # Tests: memory — follow-up question
    'How does that compare to supervised fine-tuning?',

    # Tests: search — agent design
    'What are the best practices for designing tools in a ReAct agent?',

    # Tests: out-of-scope refusal
    'What is the capital of Japan?',
]

print('=' * 65)
print('RAG AGENT SESSION — 8 Questions')
print('=' * 65)

for i, question in enumerate(TEST_QUESTIONS, 1):
    print(f'\n[Q{i}] {question}')
    print('-' * 65)

    record = ask_agent(question)

    print(f'\nANSWER: {record.answer[:300]}...')
    print(f'Tools used    : {record.tools_used}')
    print(f'Sources cited : {record.sources}')
    print(f'Faithfulness  : {record.faithfulness:.2f}')
    print(f'Relevance     : {record.relevance:.2f}')
    print(f'Latency       : {record.latency_ms:.0f}ms')
    print()

In [ ]:
# ================================================================
# STEP 7 — SESSION REPORT
# Aggregate metrics and visualisation
# ================================================================

if session_records:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Plot 1: Faithfulness + Relevance per question
    x      = range(len(session_records))
    faith  = [r.faithfulness for r in session_records]
    relev  = [r.relevance    for r in session_records]
    labels = [f'Q{i+1}' for i in x]

    axes[0].bar([i - 0.2 for i in x], faith, 0.4, label='Faithfulness', color='steelblue', alpha=0.85)
    axes[0].bar([i + 0.2 for i in x], relev, 0.4, label='Relevance',    color='tomato',    alpha=0.85)
    axes[0].axhline(0.8, color='gray', linestyle='--', alpha=0.6)
    axes[0].set_xticks(list(x))
    axes[0].set_xticklabels(labels)
    axes[0].set_ylim(0, 1.2)
    axes[0].set_title('Faithfulness & Relevance\nper Question')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')

    # Plot 2: Tool usage distribution
    tool_counts = defaultdict(int)
    for r in session_records:
        for t in r.tools_used:
            tool_counts[t] += 1
    if tool_counts:
        axes[1].bar(tool_counts.keys(), tool_counts.values(),
                    color=['steelblue','tomato','forestgreen','gold','purple'][:len(tool_counts)])
        axes[1].set_title('Tool Usage Distribution')
        axes[1].set_xticklabels(tool_counts.keys(), rotation=20, ha='right', fontsize=8)
        axes[1].grid(True, alpha=0.3, axis='y')

    # Plot 3: Latency per question
    latencies = [r.latency_ms for r in session_records]
    axes[2].bar(labels, latencies, color='mediumpurple', alpha=0.85)
    axes[2].axhline(np.mean(latencies), color='red', linestyle='--',
                    label=f'Mean: {np.mean(latencies):.0f}ms')
    axes[2].set_title('Latency per Question (ms)')
    axes[2].legend(fontsize=8)
    axes[2].grid(True, alpha=0.3, axis='y')

    plt.suptitle('RAG Agent — Session Evaluation Report', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

    # Summary table
    avg_faith = np.mean(faith)
    avg_relev = np.mean(relev)
    avg_lat   = np.mean(latencies)

    print('\nSession Summary')
    print('=' * 55)
    print(f'Questions answered     : {len(session_records)}')
    print(f'Avg faithfulness       : {avg_faith:.3f}')
    print(f'Avg relevance          : {avg_relev:.3f}')
    print(f'Avg latency            : {avg_lat:.0f}ms')
    print(f'Total tools called     : {sum(len(r.tools_used) for r in session_records)}')
    print(f'Questions with sources : {sum(1 for r in session_records if r.sources)}')
    print()

    # Per-question summary
    print(f'{"Q#":<4} {"Question":<48} {"Faith":<8} {"Relev":<8} {"Tools"}')
    print('-' * 80)
    for i, r in enumerate(session_records, 1):
        print(f'Q{i:<3} {r.question[:46]:<48} {r.faithfulness:.2f}    '
              f'{r.relevance:.2f}    {r.tools_used}')

---

## Day 18 — Capstone Summary

```
What you built:

System Architecture:
  8 source documents
  → recursive_chunker (350 tokens, 50 overlap)
  → all-MiniLM-L6-v2 embeddings
  → FAISS IndexFlatIP vector store
  → 5 agent tools (search, calculator, list, compare, learning path)
  → ReAct AgentExecutor (max 8 iterations)
  → ConversationBufferWindowMemory (k=10)
  → LLM-as-Judge evaluation (faithfulness + relevance)
  → Session report with 3 visualisation panels

Skills demonstrated:
  Day 11 : LLM API calls, temperature, token control
  Day 12 : Grounding system prompt, citation instructions, refusal
  Day 13 : SentenceTransformer embeddings, cosine similarity
  Day 14 : Chunking, VectorStore, metadata, persistence
  Day 15 : RAG prompt, faithfulness eval, context assembly
  Day 16 : LangChain LCEL, memory, AgentExecutor wrapper
  Day 17 : ReAct loop, @tool decorator, multi-tool selection

```

### Reflection Questions

1. Which tool was called most often? Does that make sense for the questions asked?
2. Which question had the lowest faithfulness score? Why?
3. The memory test (Q6: 'How does that compare...') — did the agent
   correctly use the previous conversation context?
4. The out-of-scope question (Q8) — did the agent correctly refuse?
5. If you were to deploy this in production, what would you add first?